# Azure + Databricks: Pipeline de Dados na Nuvem
## Migrando do Local para Producao

**Autora:** Nayane Araujo  
**GitHub:** [Nayanearaujo](https://github.com/Nayanearaujo)  

---

### Por que a Nuvem?

Tudo que fizemos nos notebooks anteriores rodou na sua maquina local. Isso e otimo para desenvolvimento e aprendizado. Mas em producao real, os dados sao:

- **Maiores**: em vez de 32 mil linhas, podem ser 32 milhoes
- **Em tempo real**: novos emprestimos chegam a cada segundo
- **Acessados por muitos**: multiplos times, modelos e sistemas ao mesmo tempo

Para lidar com isso, as empresas usam **servicos de nuvem** como Azure, AWS ou GCP.

### O que vamos ver neste notebook?

1. Arquitetura de nuvem para o nosso pipeline
2. Azure Data Lake Storage Gen2 (ADLS): onde guardar os dados
3. Azure Databricks: onde processar com PySpark
4. Delta Lake: formato de dados moderno (substitui CSV na nuvem)
5. Codigo PySpark equivalente ao que fizemos com Pandas
6. Azure Machine Learning: registro e deploy do modelo
7. Guia de como criar a conta gratuita e comecar

---

> **Nota**: Este notebook serve tanto para rodar no Databricks quanto para estudo.  
> As celulas marcadas com `[DATABRICKS]` devem ser executadas no ambiente do Databricks.  
> As celulas marcadas com `[LOCAL]` funcionam na sua maquina tambem.

## 1. Arquitetura Azure para o Projeto

Veja como o nosso pipeline fica quando migrado para o Azure:

In [ ]:
# [LOCAL] Diagrama da arquitetura usando plotly
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

fig = go.Figure()

# Nos do diagrama
nos = [
    # (x, y, label, cor)
    (0.5, 0.9, 'Fontes de Dados\n(Kaggle, API BCB)', '#3498db'),
    (0.5, 0.7, 'Azure Data Factory\n(Ingestao e Orquestracao)', '#9b59b6'),
    (0.2, 0.5, 'ADLS Gen2\n(Bronze Layer)', '#cd6155'),
    (0.5, 0.5, 'ADLS Gen2\n(Silver Layer)', '#e67e22'),
    (0.8, 0.5, 'ADLS Gen2\n(Gold Layer)', '#f1c40f'),
    (0.5, 0.3, 'Azure Databricks\n(PySpark + Delta Lake)', '#1abc9c'),
    (0.2, 0.1, 'Azure ML\n(Modelo de ML)', '#2ecc71'),
    (0.8, 0.1, 'Power BI / Streamlit\n(Dashboard)', '#3498db'),
]

for x, y, label, cor in nos:
    fig.add_trace(go.Scatter(
        x=[x], y=[y],
        mode='markers+text',
        marker=dict(size=60, color=cor, opacity=0.85),
        text=[label],
        textposition='middle center',
        textfont=dict(size=9, color='white'),
        showlegend=False
    ))

# Setas (conexoes)
setas = [(0.5, 0.86, 0.5, 0.74), (0.5, 0.66, 0.2, 0.54),
         (0.5, 0.66, 0.5, 0.54), (0.5, 0.66, 0.8, 0.54),
         (0.2, 0.46, 0.5, 0.34), (0.5, 0.46, 0.5, 0.34),
         (0.8, 0.46, 0.5, 0.34), (0.5, 0.26, 0.2, 0.14),
         (0.5, 0.26, 0.8, 0.14)]

for x0, y0, x1, y1 in setas:
    fig.add_annotation(
        x=x1, y=y1, ax=x0, ay=y0,
        xref='x', yref='y', axref='x', ayref='y',
        arrowhead=2, arrowsize=1.5, arrowwidth=2, arrowcolor='#7f8c8d'
    )

fig.update_layout(
    title=dict(text='Arquitetura Azure: Pipeline de Risco de Credito',
               font=dict(size=16)),
    xaxis=dict(visible=False, range=[-0.1, 1.1]),
    yaxis=dict(visible=False, range=[-0.05, 1.05]),
    height=650,
    plot_bgcolor='#f8f9fa'
)
fig.show()

## 2. Azure Data Lake Storage Gen2 (ADLS)

O ADLS e o local de armazenamento dos dados na nuvem. E como um HD externo gigante na internet, organizado em containers e pastas.

Nossa estrutura de Arquitetura Medallion fica assim no ADLS:

```
adls://credit-risk-storage/
   bronze/
      credit_risk_raw/
         year=2026/month=09/credit_risk_raw.parquet
   silver/
      credit_risk_clean/
         _delta_log/      <- logs do Delta Lake
         part-00000.parquet
   gold/
      fact_loans/
      dim_borrower/
      agg_default_by_grade/
```

Repare que usamos **Parquet** em vez de CSV. Parquet e um formato colunar que:
- Ocupa 5x a 10x menos espaco que CSV
- E lido muito mais rapido (carrega apenas as colunas necessarias)
- Preserva os tipos de dados corretamente

In [ ]:
# [LOCAL] Como acessar o ADLS via Python (azure-storage-blob)
# Para rodar localmente, instale: pip install azure-storage-blob

# Codigo de referencia - necessita de credenciais Azure reais para executar

CODIGO_ADLS = '''
from azure.storage.blob import BlobServiceClient
import pandas as pd
import io

# Configuracao de conexao
CONTA_STORAGE = "creditriskstorageXXX"
CHAVE_ACESSO  = "sua-chave-de-acesso-do-azure"
CONTAINER     = "dados-credito"

# Cria cliente do Azure Blob
client = BlobServiceClient(
    account_url=f"https://{CONTA_STORAGE}.blob.core.windows.net",
    credential=CHAVE_ACESSO
)

def upload_para_bronze(df: pd.DataFrame, nome_arquivo: str):
    """Envia um DataFrame para a camada Bronze no ADLS."""
    container_client = client.get_container_client(CONTAINER)
    
    # Converte para Parquet em memoria (sem salvar no disco)
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    buffer.seek(0)
    
    # Caminho com particao por data (boas praticas)
    from datetime import date
    hoje = date.today().strftime("%Y/%m/%d")
    caminho = f"bronze/{nome_arquivo}/{hoje}/{nome_arquivo}.parquet"
    
    blob_client = container_client.get_blob_client(caminho)
    blob_client.upload_blob(buffer, overwrite=True)
    print(f"Arquivo enviado para: {caminho}")

def ler_da_silver(nome_tabela: str) -> pd.DataFrame:
    """Le uma tabela da camada Silver do ADLS."""
    container_client = client.get_container_client(CONTAINER)
    blob = container_client.get_blob_client(f"silver/{nome_tabela}/latest.parquet")
    
    data = blob.download_blob().readall()
    df = pd.read_parquet(io.BytesIO(data))
    print(f"Lida tabela {nome_tabela}: {df.shape}")
    return df
'''

print('Codigo de referencia para conexao com ADLS:')
print(CODIGO_ADLS)
print('\nPara usar:')
print('  1. Crie uma conta de armazenamento no Azure Portal')
print('  2. Copie a chave de acesso em: Conta -> Chaves de Acesso')
print('  3. Substitua as variaveis CONTA_STORAGE e CHAVE_ACESSO')
print('  4. Use o Storage Explorer para visualizar os dados')

## 3. Azure Databricks com PySpark

O Databricks e um ambiente de processamento de dados na nuvem baseado no Apache Spark. Enquanto o Pandas processa dados na sua maquina (limitado pela RAM), o Spark distribui o processamento em dezenas de maquinas em paralelo.

O codigo PySpark e muito parecido com Pandas, mas escala para bilhoes de linhas.

**Bronze -> Silver no Databricks (PySpark)**

Veja como o nosso `silver_transform.py` ficaria em PySpark:

In [ ]:
# [DATABRICKS] Execute este codigo em um notebook do Databricks
# Substitui o arquivo silver_transform.py para escala na nuvem

CODIGO_SPARK_SILVER = '''
# Importacoes PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType, StringType
from delta.tables import DeltaTable

# Inicializa a sessao Spark (no Databricks ja existe como "spark")
# spark = SparkSession.builder.appName("CreditRiskPipeline").getOrCreate()

# Caminhos no ADLS (montado no Databricks como /mnt/credit-risk)
BRONZE_PATH = "/mnt/credit-risk/bronze/credit_risk_raw"
SILVER_PATH = "/mnt/credit-risk/silver/credit_risk_clean"

# 1. Leitura da camada Bronze
df_bronze = spark.read.parquet(BRONZE_PATH)
print(f"Bronze: {df_bronze.count():,} registros")

# 2. Remove duplicatas
df = df_bronze.dropDuplicates()
print(f"Apos remocao de duplicatas: {df.count():,}")

# 3. Tratamento de nulos
# Mediana por grade para taxa de juros
mediana_por_grade = df.groupBy("loan_grade").agg(
    F.percentile_approx("loan_int_rate", 0.5).alias("mediana_taxa")
)
df = df.join(mediana_por_grade, on="loan_grade", how="left")
df = df.withColumn(
    "loan_int_rate",
    F.coalesce(F.col("loan_int_rate"), F.col("mediana_taxa"))
).drop("mediana_taxa")

# Mediana global para tempo de emprego
mediana_emprego = df.approxQuantile("person_emp_length", [0.5], 0.01)[0]
df = df.fillna({"person_emp_length": mediana_emprego})

# 4. Validacao de ranges
df = df.filter(
    (F.col("person_age") >= 18) &
    (F.col("person_age") <= 100) &
    (F.col("person_income") > 0)
)

# 5. Padronizacao de strings
str_cols = ["person_home_ownership", "loan_intent", "loan_grade"]
for col in str_cols:
    df = df.withColumn(col, F.upper(F.trim(F.col(col))))

# 6. Adiciona metadados
from datetime import datetime
df = df.withColumn("_source", F.lit("kaggle_credit_risk"))
df = df.withColumn("_silver_timestamp", F.lit(str(datetime.now())))

# 7. Salva na Silver como Delta Lake (suporta ACID, versionamento e time travel)
df.write \\
    .format("delta") \\
    .mode("overwrite") \\
    .option("mergeSchema", "true") \\
    .save(SILVER_PATH)

print(f"Silver salva em Delta Lake: {SILVER_PATH}")
print(f"Total de registros: {df.count():,}")

# Registra como tabela no catalogo do Databricks
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS credit_risk.silver_credit_risk
    USING DELTA LOCATION \'{SILVER_PATH}\'
""")
print("Tabela registrada no catalogo: credit_risk.silver_credit_risk")
'''

print(CODIGO_SPARK_SILVER)

In [ ]:
# [DATABRICKS] Transformacao Gold com PySpark

CODIGO_SPARK_GOLD = '''
from pyspark.sql import functions as F

SILVER_PATH = "/mnt/credit-risk/silver/credit_risk_clean"
GOLD_PATH   = "/mnt/credit-risk/gold"

# Leitura da Silver
df_silver = spark.read.format("delta").load(SILVER_PATH)

# Dimensao Tomador (dim_borrower)
dim_borrower = df_silver.select(
    F.monotonically_increasing_id().alias("borrower_id"),
    "person_age", "person_income",
    "person_home_ownership", "person_emp_length",
    "cb_person_default_on_file", "cb_person_cred_hist_length"
).withColumn(
    "age_group",
    F.when(F.col("person_age") < 26, "18-25")
     .when(F.col("person_age") < 36, "26-35")
     .when(F.col("person_age") < 46, "36-45")
     .when(F.col("person_age") < 61, "46-60")
     .otherwise("60+")
).withColumn(
    "has_prior_default",
    F.when(F.col("cb_person_default_on_file") == "Y", 1).otherwise(0)
)

dim_borrower.write.format("delta").mode("overwrite") \\
    .save(f"{GOLD_PATH}/dim_borrower")

# Agregacao: taxa de inadimplencia por grade
agg_grade = df_silver.groupBy("loan_grade").agg(
    F.count("*").alias("total_emprestimos"),
    F.sum("loan_status").alias("inadimplentes"),
    F.avg("loan_status").alias("taxa_inadimplencia"),
    F.avg("loan_int_rate").alias("taxa_juros_media"),
    F.sum("loan_amnt").alias("volume_total")
).orderBy("loan_grade")

agg_grade.write.format("delta").mode("overwrite") \\
    .save(f"{GOLD_PATH}/agg_default_by_grade")

print("Camada Gold salva no Delta Lake!")
agg_grade.show()
'''

print('Transformacao Gold em PySpark:')
print(CODIGO_SPARK_GOLD)

## 4. Delta Lake: O Formato Moderno de Dados

O Delta Lake e um formato de armazenamento de dados aberto que adiciona funcionalidades essenciais ao Parquet:

- **ACID Transactions**: garantia de que operacoes de escrita nao corrompem os dados
- **Time Travel**: voce pode ler versoes antigas dos dados (DeltaTable.restoreToVersion)
- **Schema Evolution**: adicione colunas sem quebrar leituras existentes
- **Upsert (Merge)**: atualiza dados existentes sem reescrever tudo

Veja como funciona o **Time Travel** (uma das funcionalidades mais poderosas):

In [ ]:
# [DATABRICKS] Time Travel no Delta Lake

CODIGO_TIME_TRAVEL = '''
# Ler a versao atual dos dados
df_atual = spark.read.format("delta").load("/mnt/credit-risk/silver/credit_risk_clean")

# Ler como os dados estavam ontem (versao 1)
df_ontem = spark.read.format("delta") \\
    .option("versionAsOf", 1) \\
    .load("/mnt/credit-risk/silver/credit_risk_clean")

# Ler como estavam numa data especifica
df_semana_passada = spark.read.format("delta") \\
    .option("timestampAsOf", "2026-09-01") \\
    .load("/mnt/credit-risk/silver/credit_risk_clean")

# Ver historico de versoes
from delta.tables import DeltaTable
delta = DeltaTable.forPath(spark, "/mnt/credit-risk/silver/credit_risk_clean")
delta.history().show(5, truncate=False)

# Restaurar para versao anterior (rollback)
# delta.restoreToVersion(0)
'''

print('Time Travel no Delta Lake:')
print(CODIGO_TIME_TRAVEL)

print('\nBeneficios do Delta Lake para times de dados:')
print('  1. Se um processo de ETL der errado, voce volta para a versao anterior')
print('  2. Da para comparar dados de ontem com os de hoje')
print('  3. Auditoria completa de quem mudou o que e quando')
print('  4. Reproducibilidade: treinar o modelo com os dados de uma data especifica')

## 5. Azure Machine Learning: Deploy do Modelo

O Azure Machine Learning (AML) e o servico para registrar, versionar e fazer deploy de modelos de ML na nuvem.

Com o AML voce pode:
- Registrar o modelo com versao (v1, v2, v3...)
- Fazer deploy como API REST (endpoint) para predicao em tempo real
- Monitorar a performance do modelo em producao
- Configurar re-treinamento automatico quando a performance cai

In [ ]:
# [LOCAL/DATABRICKS] Registro do modelo no Azure ML
# Instale: pip install azure-ai-ml

CODIGO_AML = '''
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.identity import DefaultAzureCredential
import pickle

# Autenticacao no Azure
credential = DefaultAzureCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id="sua-subscription-id",
    resource_group_name="credit-risk-rg",
    workspace_name="credit-risk-ml-workspace"
)

# Registra o modelo (arquivo .pkl que salvamos no Notebook 03)
modelo_aml = Model(
    path="src/models/best_model.pkl",
    name="credit-risk-xgboost",
    description="Modelo XGBoost para predicao de inadimplencia de credito",
    tags={
        "autora": "Nayane Araujo",
        "dataset": "Kaggle Credit Risk Dataset",
        "auc_roc": "0.93",
        "versao": "1.0"
    }
)
modelo_registrado = ml_client.models.create_or_update(modelo_aml)
print(f"Modelo registrado: {modelo_registrado.name} v{modelo_registrado.version}")

# Cria endpoint para predicao em tempo real
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment

endpoint = ManagedOnlineEndpoint(
    name="credit-risk-endpoint",
    description="Endpoint para predicao de risco de credito em tempo real"
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

deployment = ManagedOnlineDeployment(
    name="xgboost-deployment",
    endpoint_name="credit-risk-endpoint",
    model=modelo_registrado.id,
    instance_type="Standard_DS3_v2",
    instance_count=1
)
ml_client.online_deployments.begin_create_or_update(deployment).result()
print("Endpoint criado! Acesse via API REST.")
'''

print('Registro e deploy do modelo no Azure ML:')
print(CODIGO_AML)

In [ ]:
# [LOCAL] Como chamar o endpoint do Azure ML para predicao

CODIGO_API = '''
import requests
import json

# Dados de um novo cliente
novo_cliente = {
    "person_age": 28,
    "person_income": 45000,
    "person_emp_length": 3,
    "loan_amnt": 8000,
    "loan_int_rate": 14.5,
    "loan_percent_income": 0.18,
    "loan_grade": "C",
    "loan_intent": "PERSONAL",
    "person_home_ownership": "RENT",
    "cb_person_default_on_file": "N",
    "cb_person_cred_hist_length": 4
}

# Chamada para o endpoint do Azure ML
url = "https://credit-risk-endpoint.brazilsouth.inference.ml.azure.com/score"
headers = {
    "Content-Type": "application/json",
    "Authorization": "Bearer SUA_CHAVE_AQUI"
}

response = requests.post(url, json={"data": [novo_cliente]}, headers=headers)
resultado = response.json()

prob_inadimplencia = resultado["probabilidade"][0]
print(f"Probabilidade de inadimplencia: {prob_inadimplencia:.1%}")
print(f"Decisao: {\"APROVAR\" if prob_inadimplencia < 0.5 else \"REJEITAR\"}")
'''

print('Como usar o modelo via API REST:')
print(CODIGO_API)

## 6. Como Configurar o Ambiente Azure (Passo a Passo)

Boa noticia: voce pode criar uma conta gratuita no Azure com R$1.000,00 de creditos por 12 meses!

In [ ]:
guia_azure = """
GUIA: Como Criar o Ambiente Azure para este Projeto
====================================================

Passo 1: Criar Conta Azure Gratuita
  - Acesse: https://azure.microsoft.com/pt-br/free
  - Voce recebe USD 200 de credito por 30 dias (ou R$1.000 por 12 meses)
  - Precisara de um cartao de credito (nao cobra automaticamente)

Passo 2: Criar o Resource Group
  Via CLI (instale Azure CLI primeiro):
  az login
  az group create --name credit-risk-rg --location brazilsouth

Passo 3: Criar a Conta de Armazenamento (ADLS Gen2)
  az storage account create \\
    --name creditriskstorageXXX \\
    --resource-group credit-risk-rg \\
    --location brazilsouth \\
    --sku Standard_LRS \\
    --enable-hierarchical-namespace true

Passo 4: Criar o Workspace do Databricks
  - No Azure Portal, busque por "Azure Databricks"
  - Crie um workspace no nivel "Standard" (mais barato)
  - Regiao: Brazil South
  - Apos criar, clique em "Launch Workspace"

Passo 5: Configurar o Cluster Databricks
  - Em Databricks, va em Compute -> Create Cluster
  - Runtime: 14.x LTS (ou superior) com ML
  - Tipo: Standard_DS3_v2 (4 cores, 14GB RAM)
  - Auto-terminate: 30 minutos (para nao gastar creditos)

Passo 6: Montar o ADLS no Databricks
  Em um notebook Databricks:
  dbutils.fs.mount(
      source="wasbs://dados-credito@creditriskstorageXXX.blob.core.windows.net",
      mount_point="/mnt/credit-risk",
      extra_configs={"fs.azure.account.key.creditriskstorageXXX.blob.core.windows.net":
                     "sua-chave"}
  )

Passo 7: Instalar pacotes no Cluster
  Em Libraries, instale:
  - xgboost
  - scikit-learn
  - imbalanced-learn
  - loguru

Passo 8: Importar os notebooks
  - Em Databricks Workspace, importe os notebooks desta pasta
  - Adapte os caminhos de BRONZE_PATH, SILVER_PATH, GOLD_PATH
  - Execute na ordem: Bronze -> Silver -> Gold -> ML

Custo estimado (com creditos gratuitos):
  - Cluster Standard_DS3_v2: ~R$ 0.80/hora
  - Armazenamento 10GB: ~R$ 2.00/mes
  - Para desenvolvimento, mantenha o auto-terminate ativado!
"""

print(guia_azure)

## 7. Comparativo: Local vs Nuvem

Quando usar cada um?

In [ ]:
import pandas as pd

comparativo = pd.DataFrame({
    'Aspecto': [
        'Volume de dados',
        'Custo inicial',
        'Escalabilidade',
        'Velocidade de setup',
        'Colaboracao de times',
        'Seguranca dos dados',
        'Linguagem usada',
        'Quando usar'
    ],
    'Local (Pandas + DuckDB)': [
        'Ate ~10 milhoes de linhas',
        'Zero (usa sua maquina)',
        'Limitada pela sua RAM',
        'Imediata',
        'Dificulta (arquivos locais)',
        'Menor (dados no seu HD)',
        'Python + SQL',
        'Desenvolvimento e aprendizado'
    ],
    'Nuvem (Azure + Databricks)': [
        'Bilhoes de linhas (escala infinita)',
        'A partir de R$0.80/hora (cluster)',
        'Escala com um clique',
        '30-60 minutos para configurar',
        'Facil (tudo no cloud)',
        'Alta (criptografia, controle de acesso)',
        'PySpark + SQL + Delta Lake',
        'Producao e grandes volumes'
    ]
})

print(comparativo.to_string(index=False))

In [ ]:
print('=' * 65)
print('RESUMO: PIPELINE END-TO-END COMPLETO')
print('=' * 65)

print("""
LOCAL (ja implementado):
  Bronze: src/ingestion/ingest_kaggle.py
          src/ingestion/ingest_bcb_api.py
  Silver: src/transformation/silver_transform.py
  Gold  : src/transformation/gold_transform.py
  ML    : src/models/train_model.py
          src/models/evaluate_model.py
  UI    : src/dashboard/app.py (Streamlit)

NUVEM (codigo de referencia neste notebook):
  Bronze: Azure Data Factory (ingestao automatizada)
  Silver: Databricks PySpark + Delta Lake
  Gold  : Databricks PySpark + Delta Lake
  ML    : Azure Machine Learning (registro + endpoint)
  UI    : Power BI conectado ao Gold Layer
          ou Streamlit hospedado no Azure App Service

A MESMA LOGICA DE NEGOCIO, escalando do laptop para a nuvem!
""")

print('Projeto finalizado!')
print('Nayane Araujo | github.com/Nayanearaujo')
print('=' * 65)

---

## Parabens!

Voce concluiu o projeto end-to-end completo:

- Notebook 01: Analise Exploratoria de Dados
- Notebook 02: Feature Engineering e preparacao
- Notebook 03: Treinamento e avaliacao dos modelos
- Notebook 04: Azure e Databricks (escalando para nuvem)

Este projeto demonstra todas as skills exigidas pelas vagas da EloGroup e Bancorbras.

**Nayane Araujo** | [GitHub](https://github.com/Nayanearaujo)